In [ ]:
"""
=============================================================================
 ÉTUDE 2B — M2 COMME COMPOSANTE ADDITIONNELLE DE LA FORMULE V5
=============================================================================
 L'Étude 2 a comparé M2 et M3 en tant que sources CONCURRENTES du signal
 calcique. L'Étude 2B pose une question différente : M2 apporte-t-il une
 information COMPLÉMENTAIRE, en s'ajoutant à M3 plutôt qu'en le remplaçant ?

 Elle documente également la genèse de la formule déployée, en retraçant
 le balayage de prob_cardio qui a conduit de V4 à V5.

 Entrée : resultats_phase4_v5.csv
 GPU    : non nécessaire
=============================================================================
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
CSV_PATH = '/kaggle/input/datasets/abdallahouahmane/resultas-phase4/resultats_phase4_v5.csv'

W = {'ctr': 0.15, 'cong': 0.40, 'calcif': 0.10, 'cardio': 0.35}
SEUIL_SIGNIFICATIF = 0.005     # écart d'AUC jugé négligeable en deçà

print("=" * 62)
print("  ÉTUDE 2B — M2 EN COMPOSANTE ADDITIONNELLE")
print("=" * 62)

df = pd.read_csv(CSV_PATH)
df['y_true'] = df['label_reel'].apply(
    lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0)
df['m2_norm'] = (df['calcif_pct'] / 2.0).clip(0, 1)

y        = df['y_true']
y_cardio = df['label_reel'].str.contains('Cardiomegaly', na=False).astype(int)
y_edema  = df['label_reel'].str.contains('Edema', na=False).astype(int)

print(f"\n  CSV chargé : {len(df)} images")
print(f"  Positifs : {y.sum()} | Négatifs : {(y == 0).sum()}")


def score_v5():
    return (W['ctr'] * df['s_ctr'] + W['cong'] * df['s_cong']
            + W['calcif'] * df['s_calcif'] + W['cardio'] * df['prob_cardio'])


def score_v5_plus_m2(w_m2):
    """V5 dont les quatre poids sont réduits proportionnellement
    pour libérer un poids w_m2 attribué à M2."""
    r = 1 - w_m2
    return ((W['ctr'] * r) * df['s_ctr'] + (W['cong'] * r) * df['s_cong']
            + (W['calcif'] * r) * df['s_calcif']
            + (W['cardio'] * r) * df['prob_cardio']
            + w_m2 * df['m2_norm'])


# ============================================================
# 1. FORMULES DE RÉFÉRENCE
# ============================================================
print(f"\n{'='*62}")
print("  FORMULES DE RÉFÉRENCE")
print(f"{'='*62}\n")

references = {
    'Fusion uniforme (50/25/25)':
        0.50 * df['s_ctr'] + 0.25 * df['s_calcif'] + 0.25 * df['s_cong'],
    'Pipeline V3 (45/55)':
        0.45 * df['s_ctr'] + 0.55 * df['s_cong'],
    'Optimum V4 (35/45/20)':
        0.35 * df['s_ctr'] + 0.45 * df['s_cong'] + 0.20 * df['s_calcif'],
    'Pipeline V5 (déployé)':
        score_v5(),
}

auc_ref = {}
for nom, s in references.items():
    auc_ref[nom] = roc_auc_score(y, s)
    print(f"  {nom:<30} AUC = {auc_ref[nom]:.4f}")

auc_v3 = auc_ref['Pipeline V3 (45/55)']
auc_v4 = auc_ref['Optimum V4 (35/45/20)']
auc_v5 = auc_ref['Pipeline V5 (déployé)']

# ============================================================
# 2. BALAYAGE — M2 AJOUTÉ À V5
# ============================================================
print(f"\n{'='*62}")
print("  BALAYAGE — POIDS ACCORDÉ À M2 DANS V5")
print(f"{'='*62}\n")
print(f"  {'w_m2':>7} {'w_ctr':>8} {'w_cong':>8} {'w_calcif':>9} "
      f"{'w_cardio':>9} {'AUC':>9} {'Δ vs V5':>9}")
print("  " + "-" * 62)

balayage = {}
for w_m2 in np.arange(0.00, 0.35, 0.05):
    w_m2 = round(w_m2, 2)
    r = 1 - w_m2
    s = score_v5_plus_m2(w_m2)
    a = roc_auc_score(y, s)
    balayage[w_m2] = {'auc': a, 'delta': a - auc_v5, 'scores': s}
    marque = '  ← V5' if w_m2 == 0 else ''
    print(f"  {w_m2:>7.2f} {W['ctr']*r:>8.3f} {W['cong']*r:>8.3f} "
          f"{W['calcif']*r:>9.3f} {W['cardio']*r:>9.3f} "
          f"{a:>9.4f} {a - auc_v5:>+9.4f}{marque}")

w_best = max(balayage, key=lambda k: balayage[k]['auc'])
auc_best = balayage[w_best]['auc']
gain_m2 = auc_best - auc_v5

print(f"\n  Meilleur poids : w_m2 = {w_best:.2f}  (AUC = {auc_best:.4f})")
print(f"  Gain de M2 sur V5 : {gain_m2:+.4f}")

if gain_m2 > SEUIL_SIGNIFICATIF:
    print(f"\n  M2 apporte un gain supérieur au seuil de "
          f"{SEUIL_SIGNIFICATIF:.3f} : son intégration mérite examen.")
else:
    print(f"\n  Le gain reste inférieur au seuil de "
          f"{SEUIL_SIGNIFICATIF:.3f} : M2 n'est pas retenu")
    print(f"  dans la formule de score et conserve un rôle de visualisation.")

# ============================================================
# 3. GENÈSE DE V5 — BALAYAGE DE prob_cardio
# ============================================================
print(f"\n{'='*62}")
print("  GENÈSE DE V5 — INTRODUCTION DE prob_cardio DANS V4")
print(f"{'='*62}\n")
print("  L'analyse par pathologie a révélé que V4 était anti-corrélée")
print("  à la cardiomégalie. Le balayage ci-dessous retrace la correction.\n")
print(f"  {'w_cardio':>9} {'AUC global':>12} {'AUC cardiomégalie':>19}")
print("  " + "-" * 44)

genese = {}
for w in np.arange(0.00, 0.55, 0.05):
    w = round(w, 2)
    r = 1 - w
    s = ((0.35 * r) * df['s_ctr'] + (0.45 * r) * df['s_cong']
         + (0.20 * r) * df['s_calcif'] + w * df['prob_cardio'])
    a_g, a_c = roc_auc_score(y, s), roc_auc_score(y_cardio, s)
    genese[w] = {'global': a_g, 'cardio': a_c}
    print(f"  {w:>9.2f} {a_g:>12.4f} {a_c:>19.4f}")

print(f"\n  Sans prob_cardio, l'AUC sur la cardiomégalie est de "
      f"{genese[0.0]['cardio']:.4f},")
print(f"  soit très en deçà de 0.50 : le score décroît lorsque la")
print(f"  cardiomégalie est présente. L'ajout de cette composante")
print(f"  corrige l'inversion et améliore simultanément l'AUC globale.")

# ============================================================
# 4. AUC PAR PATHOLOGIE — V3, V4, V5
# ============================================================
print(f"\n{'='*62}")
print("  AUC PAR PATHOLOGIE SELON LA FORMULE")
print(f"{'='*62}\n")
print(f"  {'Formule':<26} {'Global':>9} {'Cardiomég.':>12} {'Œdème':>9}")
print("  " + "-" * 58)
for nom, s in references.items():
    print(f"  {nom:<26} {roc_auc_score(y, s):>9.4f} "
          f"{roc_auc_score(y_cardio, s):>12.4f} "
          f"{roc_auc_score(y_edema, s):>9.4f}")

print(f"\n  Le passage de V3 à V5 relève l'AUC sur la cardiomégalie de")
print(f"  {roc_auc_score(y_cardio, references['Pipeline V3 (45/55)']):.4f} à "
      f"{roc_auc_score(y_cardio, references['Pipeline V5 (déployé)']):.4f}, "
      f"au prix d'une baisse sur l'œdème")
print(f"  de {roc_auc_score(y_edema, references['Pipeline V3 (45/55)']):.4f} à "
      f"{roc_auc_score(y_edema, references['Pipeline V5 (déployé)']):.4f}. "
      f"L'équilibre entre les deux pathologies")
print(f"  est amélioré, ce qui explique le gain sur l'AUC globale.")

# ============================================================
# 5. VISUALISATION
# ============================================================
fig = plt.figure(figsize=(21, 13))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

w_list = list(balayage.keys())
a_list = [balayage[w]['auc'] for w in w_list]
d_list = [balayage[w]['delta'] for w in w_list]

# ── Panel 1 : AUC en fonction de w_m2 ─────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(w_list, a_list, '-o', color='#185FA5', lw=2.5, ms=8,
         label='V5 + M2')
ax1.axhline(auc_v5, color='#C0392B', ls='--', lw=2,
            label=f'V5 sans M2 ({auc_v5:.4f})')
ax1.fill_between(w_list, auc_v5, a_list,
                 where=[a >= auc_v5 for a in a_list],
                 alpha=0.18, color='green')
ax1.fill_between(w_list, auc_v5, a_list,
                 where=[a < auc_v5 for a in a_list],
                 alpha=0.18, color='red')
ax1.set_xlabel('Poids accordé à M2', fontsize=11)
ax1.set_ylabel('AUC-ROC', fontsize=11)
ax1.set_title("Étude 2B — AUC de V5 augmentée d'une composante M2\n"
              'Les quatre poids de V5 sont réduits proportionnellement',
              fontweight='bold', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
for wx, ay in zip(w_list, a_list):
    ax1.annotate(f'{ay:.4f}', (wx, ay), textcoords='offset points',
                 xytext=(0, 11), ha='center', fontsize=8)

# ── Panel 2 : évolution des formules ──────────────────────
ax2 = fig.add_subplot(gs[0, 2])
noms_ref = list(auc_ref.keys())
vals_ref = list(auc_ref.values())
bars = ax2.barh([n.split(' (')[0] for n in noms_ref], vals_ref,
                color=['#95A5A6', '#7F8C8D', '#1B2A4A', '#C0392B'], alpha=0.88)
ax2.set_xlim(min(vals_ref) - 0.03, max(vals_ref) + 0.03)
ax2.set_xlabel('AUC-ROC')
ax2.set_title('Évolution des formules\nde fusion', fontweight='bold')
ax2.grid(alpha=0.3, axis='x')
for bar, v in zip(bars, vals_ref):
    ax2.text(v + 0.002, bar.get_y() + bar.get_height() / 2,
             f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

# ── Panel 3 : genèse de V5 ────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
w_g = list(genese.keys())
ax3.plot(w_g, [genese[w]['global'] for w in w_g], '-o', color='#185FA5',
         lw=2.2, ms=5, label='AUC globale')
ax3.plot(w_g, [genese[w]['cardio'] for w in w_g], '-s', color='#C0392B',
         lw=2.2, ms=5, label='AUC cardiomégalie')
ax3.axhline(0.50, color='gray', ls=':', lw=1.2, label='Hasard')
ax3.axvline(W['cardio'], color='green', ls='--', lw=1.8,
            label=f"Poids retenu ({W['cardio']})")
ax3.set_xlabel('Poids de prob_cardio')
ax3.set_ylabel('AUC-ROC')
ax3.set_title('Genèse de V5\nCorrection de la cardiomégalie', fontweight='bold')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

# ── Panel 4 : écart induit par M2 ─────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.bar(w_list, d_list, width=0.032, alpha=0.85,
        color=['#16A34A' if d >= 0 else '#DC2626' for d in d_list])
ax4.axhline(0, color='black', lw=1.4)
ax4.axhline(SEUIL_SIGNIFICATIF, color='green', ls='--', alpha=0.6,
            label=f'Seuil de significativité ({SEUIL_SIGNIFICATIF})')
ax4.set_xlabel('Poids accordé à M2')
ax4.set_ylabel('Écart d\'AUC par rapport à V5')
ax4.set_title('Effet de M2 sur le score\nselon son poids', fontweight='bold')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3, axis='y')

# ── Panel 5 : AUC par pathologie ──────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
x = np.arange(len(references))
larg = 0.27
ax5.bar(x - larg, [roc_auc_score(y, s) for s in references.values()],
        larg, label='Global', color='#185FA5', alpha=0.85)
ax5.bar(x, [roc_auc_score(y_cardio, s) for s in references.values()],
        larg, label='Cardiomégalie', color='#C0392B', alpha=0.85)
ax5.bar(x + larg, [roc_auc_score(y_edema, s) for s in references.values()],
        larg, label='Œdème', color='#F39C12', alpha=0.85)
ax5.axhline(0.50, color='gray', ls=':', lw=1.2)
ax5.set_xticks(x)
ax5.set_xticklabels([n.split(' (')[0].replace('Pipeline ', '').replace('Optimum ', '')
                     for n in references], fontsize=8, rotation=20)
ax5.set_ylabel('AUC-ROC')
ax5.set_title('AUC par pathologie\nselon la formule', fontweight='bold')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.3, axis='y')

plt.suptitle(
    f'ÉTUDE 2B — M2 en composante additionnelle | '
    f'Gain maximal {gain_m2:+.4f} | V5 : AUC = {auc_v5:.4f} | '
    f'n = {len(df)} images NIH',
    fontsize=13, fontweight='bold')

for ext in ('png', 'svg'):
    plt.savefig(f'etude2b_m2_additionnel.{ext}', dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

# ============================================================
# 6. SYNTHÈSE
# ============================================================
print(f"\n{'='*62}")
print("  SYNTHÈSE — ÉTUDE 2B")
print(f"{'='*62}\n")

print(f"  Question posée : la surface calcifiée segmentée par M2 apporte-t-elle")
print(f"  une information complémentaire à celle du classifieur M3 ?\n")
print(f"    Pipeline V5 sans M2      : AUC = {auc_v5:.4f}")
print(f"    Meilleure variante + M2  : AUC = {auc_best:.4f}  "
      f"(poids {w_best:.2f})")
print(f"    Gain                     : {gain_m2:+.4f}")

print(f"\n  Conclusion : le gain maximal de {gain_m2:+.4f} demeure sous le seuil")
print(f"  de significativité retenu. M2 n'entre pas dans le calcul du score ;")
print(f"  il reste employé pour la restitution visuelle des zones calcifiées,")
print(f"  où sa segmentation (Dice = 0.784) conserve toute sa valeur.")

print(f"\n  Fichiers générés : etude2b_m2_additionnel.png / .svg")